# Milestone 1 crawler on Colab (CPU)

One machine crawls the **whole** set (`id_ranges.csv`, D8). Put `S2_API_KEY` in Colab **Secrets** (never in the notebook).

**Chạy khi rảnh, dừng khi bận — không mất dữ liệu** if you follow this loop every session:

1. **Setup** (clone/pull, mount Drive, secrets) — once per session
2. **Restore** Drive → `/content/data` (`restore_now()`) — unpacks `arxiv-data.tar.gz` in seconds (or migrates legacy `arxiv-data/` folder once in parallel)
3. **Crawl** — same command every time; manifest skips finished IDs, auto-backups to `arxiv-data.tar.gz` every 300 new IDs (~35 min) and again on exit
4. When you stop: **Runtime → Interrupt**, then run **Backup** (`backup_now()` or `backup_now(unmount=True)` before closing tab)
5. Next session after disconnect: repeat from step 1

After **disconnect**, re-run from the top. If Drive mount fails (`Mountpoint must not already contain files`), use **Runtime → Restart session**, then run again (do **not** use `force_remount` on a non-empty `/content/drive`).

Data lives on Colab local disk (`/content/data`) while crawling (SQLite + Drive FUSE is unsafe). Backup packs `data/` into one atomic archive `MyDrive/arxiv-data.tar.gz` with shrink protection.

In [ ]:
import os

if not os.path.isdir("/content/arxiv-pipeline"):
    !git clone https://github.com/GiaaPhuc/arxiv-pipeline.git
else:
    !git -C /content/arxiv-pipeline pull --ff-only
%cd /content/arxiv-pipeline/src
!pip install -q -r requirements.txt

In [ ]:
import logging
import os
import sys
import time
from pathlib import Path

from google.colab import drive, userdata

if "/content/arxiv-pipeline/src" not in sys.path:
    sys.path.insert(0, "/content/arxiv-pipeline/src")

from crawler.checkpoint import (
    backup_tar_archive,
    is_workspace_complete,
    manifest_pair_counts,
    read_archive_meta,
    restore_data_dir,
)

_DRIVE_MOUNT = "/content/drive"
_DATA_DIR = Path("/content/data")
_BACKUP_TAR_GZ = Path("/content/drive/MyDrive/arxiv-data.tar.gz")
_BACKUP_TAR = Path("/content/drive/MyDrive/arxiv-data.tar")
_BACKUP_LEGACY_DIR = Path("/content/drive/MyDrive/arxiv-data")


def _drive_usable() -> bool:
    my_drive = os.path.join(_DRIVE_MOUNT, "MyDrive")
    if not os.path.isdir(my_drive):
        return False
    try:
        os.listdir(my_drive)
        return True
    except OSError:
        return False


if _drive_usable():
    print("Google Drive already mounted.")
else:
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.isdir(_DRIVE_MOUNT) and os.listdir(_DRIVE_MOUNT):
        raise RuntimeError(
            "Cannot mount Drive: /content/drive still has files "
            "(common after disconnect). Runtime → Restart session, "
            "then run this notebook from the top."
        )
    drive.mount(_DRIVE_MOUNT)

os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
os.environ["CRAWLER_CONTACT"] = "your-email@example.com"  # User-Agent contact
os.environ["CRAWLER_DATA_DIR"] = str(_DATA_DIR)
os.environ["CRAWLER_BACKUP_DIR"] = str(_BACKUP_TAR_GZ)
os.environ["CRAWLER_BACKUP_EVERY_IDS"] = "300"  # ~35 min + final backup on exit
os.environ["CRAWLER_ARXIV_MIN_INTERVAL"] = "3.5"  # Colab: small margin above 3 s


def _archive_rank(path: Path) -> tuple[int, float]:
    meta = read_archive_meta(path)
    raw_pairs = meta.get("final_pairs") if meta else None
    pairs = int(raw_pairs) if isinstance(raw_pairs, (int, float)) else -1
    return (pairs, path.stat().st_mtime)


def backup_now(*, allow_shrink: bool = False, unmount: bool = False) -> None:
    """Pack /content/data into one atomic archive on Drive (with shrink guard)."""
    if not _drive_usable():
        raise RuntimeError("Google Drive is not mounted; re-run the Setup cell first.")
    if not (_DATA_DIR / "manifest.sqlite").is_file():
        print("Nothing to back up yet (/content/data/manifest.sqlite not found).")
        return
    info = backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ, allow_shrink=allow_shrink)
    mib = info["archive_bytes"] / (1024 * 1024)
    print(
        f"backup done {time.strftime('%H:%M:%S')}: {_BACKUP_TAR_GZ.name} "
        f"({mib:.1f} MiB, {info['final_pairs']} final pairs)"
    )
    if unmount:
        drive.flush_and_unmount()
        print("Drive flushed to cloud and unmounted — safe to close Colab.")


def restore_now(*, force: bool = False) -> None:
    """Restore /content/data from Drive archive (or migrate legacy folder once)."""
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
    if not _drive_usable():
        raise RuntimeError("Google Drive is not mounted; re-run the Setup cell first.")
    if not force and is_workspace_complete(_DATA_DIR):
        local_final, _ = manifest_pair_counts(_DATA_DIR / "manifest.sqlite")
        print(
            f"Skip restore: {_DATA_DIR} already has complete local data "
            f"({local_final} final pairs). Pass restore_now(force=True) to overwrite."
        )
        return

    archives = [p for p in (_BACKUP_TAR_GZ, _BACKUP_TAR) if p.is_file()]
    if archives:
        chosen = max(archives, key=_archive_rank)
        info = restore_data_dir(chosen, _DATA_DIR, force=force)
        print(f"Restored {chosen.name} -> {_DATA_DIR} ({info['final_pairs']} final pairs).")
        if chosen != _BACKUP_TAR_GZ:
            backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ)
            print(f"Migrated {chosen.name} -> {_BACKUP_TAR_GZ.name}.")
        return

    has_legacy = _BACKUP_LEGACY_DIR.is_dir() and (
        (_BACKUP_LEGACY_DIR / "manifest.sqlite").is_file()
        or (_BACKUP_LEGACY_DIR / "papers").is_dir()
    )
    if has_legacy:
        print(
            "Legacy folder backup found on Drive: running 16-thread parallel restore "
            "and migrating to arxiv-data.tar.gz..."
        )
        info = restore_data_dir(
            _BACKUP_LEGACY_DIR,
            _DATA_DIR,
            force=force,
            legacy_workers=16,
            progress_every=250,
        )
        if (_DATA_DIR / "manifest.sqlite").is_file():
            bk = backup_tar_archive(_DATA_DIR, _BACKUP_TAR_GZ)
            mib = bk["archive_bytes"] / (1024 * 1024)
            print(
                f"Migrated legacy folder -> {_BACKUP_TAR_GZ.name} "
                f"({mib:.1f} MiB, {info['final_pairs']} final pairs). "
                "Next restore will take seconds!"
            )
        return

    _DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("No backup on Drive (first run).")

In [ ]:
# Plan only: no network.
!python run_crawl.py --ranges id_ranges.csv --dry-run

In [ ]:
# Restore from Drive — run EVERY new session (auto-skips if local data is complete).
restore_now()

In [ ]:
# Crawl: re-run the SAME active line after every restore to resume.
# Exit code 3 = arXiv still rate-limiting after waits — wait 30–60 min, try again later.
# First session — pilot 30 IDs (switch to full crawl after pilot OK):
!python run_crawl.py --ranges id_ranges.csv --limit 30
# !python run_crawl.py --ranges id_ranges.csv

### Quy trình dừng / chạy tiếp

**A. Dừng tạm nhưng runtime vẫn còn sống** (Interrupt / hết lệnh / exit code 3):
1. `run_crawl.py` đã tự chạy backup cuối vào `arxiv-data.tar.gz` trước khi thoát; muốn chắc chắn hơn thì chạy thêm cell **Backup** bên dưới (`backup_now()`).
2. Để crawl tiếp: chạy lại cell **Crawl** (không cần chạy lại Restore; nếu lỡ chạy `restore_now()` thì hàm sẽ tự bỏ qua khi `/content/data` đã đủ dữ liệu).
3. Nếu sắp đóng tab / tắt máy hẳn: chạy `backup_now(unmount=True)` để ép Colab đẩy hết cache lên Google Drive trước khi đóng.

**B. Bị disconnect / mất runtime**:
- Chạy lại từ đầu (**Setup → Restore → Crawl**). Dữ liệu khôi phục từ bản `arxiv-data.tar.gz` gần nhất trên Drive (tự lưu mỗi 300 ID mới ~35 phút và khi dừng).

In [ ]:
# Backup — run after Interrupt (use unmount=True right before closing the Colab tab).
backup_now()
# backup_now(unmount=True)

In [ ]:
# ONLY when crawl finished (no exit 3 / no pending pairs) — not after a partial run:
# !python validate_corpus.py --data-dir $CRAWLER_DATA_DIR
# !python export_data.py --data-dir $CRAWLER_DATA_DIR